# 02 - BDD100K dataset exploration
Convert BDD100K JSON labels to YOLO format and run basic EDA. Set the paths in the first cell.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BDD_ROOT = Path('/content/drive/MyDrive/bdd100k')          # edit
LABELS = {'train': BDD_ROOT / 'labels/bdd100k_labels_images_train.json',
          'val':   BDD_ROOT / 'labels/bdd100k_labels_images_val.json'}
OUT = Path('/content/bdd100k_yolo')
W, H = 1280, 720
CLASSES = ['pedestrian','rider','car','truck','bus','train','motorcycle','bicycle','traffic light','traffic sign']
CLS_ID = {c: i for i, c in enumerate(CLASSES)}

In [ ]:
rows = []
for split, path in LABELS.items():
    (OUT / 'labels' / split).mkdir(parents=True, exist_ok=True)
    data = json.load(open(path))
    for item in data:
        lines = []
        for lab in item.get('labels', []):
            cat = lab['category']
            if cat not in CLS_ID or 'box2d' not in lab:
                continue
            b = lab['box2d']
            x1, y1, x2, y2 = b['x1'], b['y1'], b['x2'], b['y2']
            xc, yc, bw, bh = (x1+x2)/2/W, (y1+y2)/2/H, (x2-x1)/W, (y2-y1)/H
            lines.append(f'{CLS_ID[cat]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
            rows.append({'split': split, 'cls': cat, 'w': x2-x1, 'h': y2-y1})
        stem = Path(item['name']).stem
        (OUT / 'labels' / split / f'{stem}.txt').write_text('\n'.join(lines))
df = pd.DataFrame(rows)
print(len(df), 'boxes')

In [ ]:
# Class distribution
counts = df.groupby(['cls', 'split']).size().unstack(fill_value=0).reindex(CLASSES)
counts.plot(kind='bar', figsize=(10, 4), logy=True, title='Boxes per class (log scale)')
plt.tight_layout(); plt.show()
counts

In [ ]:
# Box size distribution
df['area'] = df.w * df.h
df['area'].clip(upper=df['area'].quantile(0.99)).plot(kind='hist', bins=60, figsize=(8, 3), title='Box area (px^2)')
plt.show()

Next: copy the matching images into `images/train` and `images/val` under the output folder, then point `configs/data.yaml` at it.